# 🚀 تشغيل نموذج مفتوح على Google Colab (T4 مجاني)

**ما يمنحك Colab مجاناً:** بطاقة `NVIDIA T4` بسعة 16GB، جلسة تصل إلى 12 ساعة.

## قبل التشغيل
1. **Runtime** → `Change runtime type` → **Hardware accelerator: T4 GPU**
2. ثم **Runtime** → **Run all**.

> الحد 16GB يعني أن أفضل خيار هو نموذج بحجم 14B بتكميم 4-bit.
> للحصول على بطاقتين (32GB) استخدم دفتر Kaggle بدلاً من هذا.

## 1) التحقق من البطاقة

In [ ]:
!nvidia-smi

## 2) تثبيت Ollama

In [ ]:
# تثبيت Ollama (محرّك تشغيل النماذج المفتوحة)
import subprocess

subprocess.run("apt-get -qq update && apt-get -qq install -y zstd", shell=True, check=False)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
print(subprocess.run("ollama --version", shell=True, capture_output=True, text=True).stdout)

## 3) تشغيل الخادم

In [ ]:
import os, subprocess, time, urllib.request

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=os.environ,
)

for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("✅ Ollama جاهز")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("لم يبدأ Ollama")

## 4) تنزيل النموذج

In [ ]:
MODEL = "qwen3:14b"   # Colab المجاني = بطاقة T4 واحدة (16GB)
# "gemma3:12b"     -> بديل جيد
# "gpt-oss:20b"    -> من OpenAI (رخصة Apache-2.0)
# "qwen3:8b"       -> الأسرع

!ollama pull {MODEL}

## 5) اختبار سريع

In [ ]:
!ollama run {MODEL} "عرّف بنفسك باختصار بالعربية" --verbose

## 6) رابط عام للوصول من جهازك

In [ ]:
# رابط عام مؤقت (Cloudflare Tunnel) — يبقى ما دامت الجلسة تعمل
import re, subprocess, time

subprocess.run(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
    "cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
    shell=True, check=True,
)

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

PUBLIC_URL = None
for _ in range(120):
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0)
        break

if not PUBLIC_URL:
    raise RuntimeError("تعذّر إنشاء النفق")

print("🌍 الرابط العام :", PUBLIC_URL)
print("🔌 نقطة API متوافقة مع OpenAI :", PUBLIC_URL + "/v1")
print("⚠️ الرابط مؤقت وغير محمي — لا تشاركه ولا ترسله بيانات حساسة.")

## 7) واجهة دردشة داخل الدفتر (اختياري)

In [ ]:
# واجهة دردشة داخل الدفتر (تعمل عبر النقطة المحلية)
!pip -q install gradio openai

import gradio as gr
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:11434/v1", api_key="ollama")


def chat(message, history):
    messages = [{"role": "system", "content": "أنت مساعد ذكي مفيد. أجب بدقة وباختصار."}]
    for turn in history or []:
        messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})
    resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.7)
    return resp.choices[0].message.content


gr.ChatInterface(chat, title=f"دردشة مع {MODEL}").launch(share=False, inline=True)